# 23 — Screener định giá toàn sàn

**Sản phẩm 2.** Chấm điểm toàn bộ cổ phiếu HOSE trên bốn trục — định giá, khả
năng sinh lời, tăng trưởng, an toàn tài chính — rồi xếp hạng **trong từng
ngành**, không phải trên toàn thị trường.

Bốn quyết định thiết kế, mỗi cái giải quyết một cách hỏng cụ thể:

| Quyết định | Nếu không làm |
|---|---|
| Chuẩn hoá **trong ngành** | P/E ngân hàng 8 lần và P/E công nghệ 25 lần trở thành "ngân hàng rẻ hơn" |
| Ngành nhỏ dùng thang toàn thị trường | z-score trên 3 quan sát là một con số ngẫu nhiên |
| Đọc `higher_is_better` từ danh mục | P/E thấp là tốt, ROE thấp là xấu — dấu phải lấy từ dữ liệu |
| Cắt đuôi (winsorize) trước khi z-score | một mã P/E 900 lần đè bẹp cả thang của ngành |

Và một cảnh báo đặt trước mọi thứ: **đây là công cụ thu hẹp danh sách, không
phải khuyến nghị đầu tư.** Điểm cao nghĩa là "đáng đọc báo cáo", không phải
"nên mua".

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, heatmap, hom_nay, lui_ngay
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Vũ trụ: lọc thanh khoản trước

Xếp hạng trên cả 407 mã cho ra một danh sách đứng đầu bởi những mã khớp vài
trăm cổ phiếu một phiên. Định giá của chúng không sai — nó chỉ là giá của một
thị trường không tồn tại.

In [2]:
danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")

gia = client.eod.stock.ohlcv(danh_muc["symbol"].tolist(), start=lui_ngay(HOM_NAY, thang=3))
gtgd = (
    gia.assign(gtgd=gia["close"] * gia["volume"] * 1_000)  # nghìn VND → VND
    .groupby("symbol", observed=True)["gtgd"]
    .mean()
)

NGUONG_TY = 5.0
VU_TRU = gtgd[gtgd >= NGUONG_TY * 1e9].index.tolist()

print(f"{len(danh_muc)} mã HOSE → {len(VU_TRU)} mã có GTGD bình quân ≥ {NGUONG_TY:.0f} tỷ/phiên")

407 mã HOSE → 140 mã có GTGD bình quân ≥ 5 tỷ/phiên


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


## 2 · Kỳ báo cáo: năm gần nhất đã đủ bốn quý

Không viết cứng `2025`. Hỏi dữ liệu xem năm nào là năm gần nhất có số liệu đầy
đủ — notebook này phải chạy được vào tháng 3 năm sau mà không cần sửa gì.

In [3]:
thu = client.financials.indicators(
    VU_TRU[:50], codes=["roe"], period="annual", start_year=HOM_NAY.year - 2
)
do_phu = thu.groupby("year", observed=True)["symbol"].nunique()
print("Số mã có số liệu năm, theo năm:")
print(do_phu.to_string())

NAM = int(do_phu[do_phu >= do_phu.max() * 0.9].index.max())
print(f"\n→ dùng năm {NAM}")

Số mã có số liệu năm, theo năm:
year
2024    50
2025    50

→ dùng năm 2025


## 3 · Lấy chỉ tiêu và đọc chiều tốt/xấu từ danh mục

**Không tự quyết định dấu.** `higher_is_better` nằm sẵn trong
`indicator_catalog()`; viết tay bảng dấu là tạo ra một chỗ để sai lệch với
nguồn mà không ai phát hiện.

In [4]:
TRUC = {
    "Định giá": ["pe", "pb"],
    "Sinh lời": ["roe", "net_margin"],
    "Tăng trưởng": ["revenue_growth", "npat_growth"],
    "An toàn": ["debt_to_equity"],
}
MA_CHI_TIEU = [m for ds in TRUC.values() for m in ds]

cat = client.financials.indicator_catalog()
chieu = (
    cat[cat["code"].isin(MA_CHI_TIEU)]
    .groupby("code", observed=True)["higher_is_better"]
    .first()
)
print("Chiều tốt/xấu đọc từ danh mục:")
print(chieu.to_string())

assert chieu.notna().all(), "Có chỉ tiêu không xác định được chiều — không xếp hạng được"

Chiều tốt/xấu đọc từ danh mục:


code
debt_to_equity    False
net_margin         True
npat_growth        True
pb                False
pe                False
revenue_growth     True
roe                True


In [5]:
raw = client.financials.indicators(
    VU_TRU, codes=MA_CHI_TIEU, period="annual", start_year=NAM, end_year=NAM
)
bang = raw.pivot_table(index="symbol", columns="code", values="value")

loai_hinh = raw.groupby("symbol", observed=True)["company_type"].first()
bang = bang.join(loai_hinh).join(
    danh_muc.set_index("symbol")[["short_name", "icb_name2", "icb_level2"]]
)

print(f"{len(bang)} mã · độ phủ từng chỉ tiêu:")
print((bang[MA_CHI_TIEU].notna().sum() / len(bang) * 100).round(1).astype(str).add(" %").to_string())

140 mã · độ phủ từng chỉ tiêu:
pe                 98.6 %
pb                 99.3 %
roe               100.0 %
net_margin         72.9 %
revenue_growth     83.6 %
npat_growth       100.0 %
debt_to_equity     83.6 %


### ⚠️ P/E rỗng không phải là "rẻ"

Danh mục nói rõ: *"Rỗng khi lợi nhuận âm"*. Một doanh nghiệp lỗ không có P/E —
và nếu bạn `fillna(0)` thì nó thành mã rẻ nhất sàn.

In [6]:
lo = bang[bang["pe"].isna()]
print(f"{len(lo)} mã không có P/E (lợi nhuận âm hoặc thiếu dữ liệu):")
print(lo[["short_name", "icb_name2", "roe"]].head(10).to_string())

2 mã không có P/E (lợi nhuận âm hoặc thiếu dữ liệu):
           short_name             icb_name2       roe
symbol                                               
GEL     Hạ tầng GELEX  Xây dựng và Vật liệu  0.025341
KOS      Công ty KOSY          Bất động sản  0.008105


Cách xử lý: **để nguyên `NaN`**. Điểm của trục "Định giá" sẽ tính từ P/B, và mã
đó vẫn được chấm — chỉ là thiếu một cấu phần. Không bịa số, không loại bỏ im
lặng.

## 4 · Cắt đuôi rồi chuẩn hoá trong ngành

Hai bước, và thứ tự quan trọng: **cắt đuôi trước, z-score sau**. Một mã P/B 47
lần trong ngành 20 mã sẽ đẩy trung bình lên và dồn 19 mã còn lại vào một điểm.

In [7]:
NGUONG_NGANH_NHO = 8  # dưới số này thì z-score trong ngành là nhiễu


def cat_duoi(s: pd.Series, duoi: float = 0.05, tren: float = 0.95) -> pd.Series:
    """Kẹp giá trị vào khoảng phân vị — giữ nguyên số quan sát, chỉ bỏ đuôi."""
    if s.notna().sum() < 3:
        return s
    lo, hi = s.quantile(duoi), s.quantile(tren)
    return s.clip(lo, hi)


def z_score(s: pd.Series) -> pd.Series:
    """Z-score, trả về 0 cho cả nhóm nếu độ lệch chuẩn bằng 0."""
    sd = s.std()
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / sd


def chuan_hoa(df: pd.DataFrame, ma_chi_tieu: str, cot_nhom: str = "icb_level2") -> pd.Series:
    """Z-score trong ngành, có dấu theo `higher_is_better`, ngành nhỏ dùng thang toàn sàn.

    Ngành dưới `NGUONG_NGANH_NHO` mã được chuẩn hoá trên **toàn bộ vũ trụ** thay
    vì trong ngành: một z-score tính trên ba quan sát mang nhiều thông tin về
    việc "nhóm này có ba mã" hơn là về doanh nghiệp.
    """
    ket_qua = pd.Series(np.nan, index=df.index, dtype=float)
    dem = df.groupby(cot_nhom, observed=True)[ma_chi_tieu].transform("count")

    du_lon = dem >= NGUONG_NGANH_NHO
    if du_lon.any():
        ket_qua[du_lon] = (
            df[du_lon]
            .groupby(cot_nhom, observed=True)[ma_chi_tieu]
            .transform(lambda s: z_score(cat_duoi(s)))
        )
    if (~du_lon).any():
        toan_san = z_score(cat_duoi(df[ma_chi_tieu]))
        ket_qua[~du_lon] = toan_san[~du_lon]

    # Dấu lấy từ danh mục, không phải từ trí nhớ
    return ket_qua if chieu[ma_chi_tieu] else -ket_qua


for ma_ct in MA_CHI_TIEU:
    bang[f"z_{ma_ct}"] = chuan_hoa(bang, ma_ct)

nho = bang.groupby("icb_name2", observed=True)["pe"].count()
print(f"Ngành dùng thang toàn sàn (dưới {NGUONG_NGANH_NHO} mã): {sorted(nho[nho < NGUONG_NGANH_NHO].index)}")

Ngành dùng thang toàn sàn (dưới 8 mã): ['Bán lẻ', 'Bảo hiểm', 'Công nghệ Thông tin', 'Du lịch và Giải trí', 'Dầu khí', 'Hàng cá nhân & Gia dụng', 'Truyền thông', 'Tài nguyên Cơ bản', 'Y tế', 'Ô tô và phụ tùng', 'Điện, nước & xăng dầu khí đốt']

## 5 · Điểm từng trục và điểm tổng

Điểm mỗi trục là **trung bình các z-score có mặt** — mã thiếu một chỉ tiêu vẫn
được chấm trên chỉ tiêu còn lại thay vì bị loại. Điểm tổng là trung bình bốn
trục, cũng chỉ tính trên trục có dữ liệu.

In [8]:
for ten_truc, ma_ds in TRUC.items():
    bang[ten_truc] = bang[[f"z_{m}" for m in ma_ds]].mean(axis=1, skipna=True)

bang["Điểm"] = bang[list(TRUC)].mean(axis=1, skipna=True)
bang["số trục có dữ liệu"] = bang[list(TRUC)].notna().sum(axis=1)

# Yêu cầu tối thiểu ba trục — hai trục không đủ để gọi là một đánh giá
ket_qua = bang[bang["số trục có dữ liệu"] >= 3].sort_values("Điểm", ascending=False)

print(f"{len(ket_qua)}/{len(bang)} mã được chấm trên ít nhất 3 trục")
print(f"Điểm: từ {ket_qua['Điểm'].min():.2f} tới {ket_qua['Điểm'].max():.2f}")

140/140 mã được chấm trên ít nhất 3 trục
Điểm: từ -1.89 tới 1.72


In [9]:
COT_HIEN = ["short_name", "icb_name2", "Điểm", *TRUC, "pe", "pb", "roe", "npat_growth"]

top20 = ket_qua.head(20)[COT_HIEN].copy()
top20["roe"] = (top20["roe"] * 100).round(1)
top20["npat_growth"] = (top20["npat_growth"] * 100).round(1)
top20.round(2).rename(columns={"roe": "roe %", "npat_growth": "LNST tăng %"})

,short_name,icb_name2,Điểm,Định giá,Sinh lời,Tăng trưởng,An toàn,pe,pb,roe %,LNST tăng %
symbol,,,,,,,,,,,
VIX,Chứng khoán VIX,Dịch vụ tài chính,1.72,0.89,2.14,2.68,1.19,6.37,1.61,28.9,715.6
HHS,Đầu tư DV Hoàng Huy,Ô tô và phụ tùng,1.55,1.03,1.92,2.27,0.98,1.50,0.33,21.6,867.7
KLB,KienlongBank,Ngân hàng,1.22,0.73,1.54,1.40,NaN,5.23,1.15,24.7,109.1
ANV,Thủy sản Nam Việt,Thực phẩm và đồ uống,1.07,0.56,0.93,2.14,0.67,6.85,1.94,28.3,1989.6
HDC,Phát triển Nhà Bà Rịa Vũng Tàu,Bất động sản,1.02,0.58,2.37,0.87,0.28,7.07,1.54,21.8,867.5
KSB,Khoáng sản Bình Dương,Xây dựng và Vật liệu,0.98,0.27,0.40,1.94,1.30,12.68,0.70,5.6,193.7
VCG,VINACONEX,Xây dựng và Vật liệu,0.96,0.81,1.47,1.20,0.35,3.54,1.17,29.4,295.6
HAH,Vận tải và Xếp dỡ Hải An,Hàng & Dịch vụ Công nghiệp,0.93,0.56,1.25,1.14,0.79,7.08,1.84,22.4,85.5
NT2,Điện lực Nhơn Trạch 2,"Điện, nước & xăng dầu khí đốt",0.92,0.60,0.68,1.93,0.46,7.01,1.44,23.3,1263.5


## 6 · Đọc kết quả

### Điểm bốn trục của top 15 — nhìn được điểm mạnh yếu chứ không chỉ thứ hạng

In [10]:
top15 = ket_qua.head(15)
bang_truc = top15[list(TRUC)].round(2)
bang_truc.index = top15["short_name"].str.slice(0, 22)

heatmap(
    bang_truc,
    tieu_de="Điểm bốn trục — 15 mã dẫn đầu",
    phu_de="Z-score trong ngành · dương là tốt hơn trung bình ngành",
    nhan_mau="z-score",
    dinh_dang_o="%{z:+.2f}",
)

Bảng này quan trọng hơn thứ hạng. Một mã điểm tổng cao nhờ ba trục tốt và một
trục rất xấu là một hồ sơ khác hẳn một mã đều bốn trục — và điểm tổng gộp cả
hai vào một con số giống nhau.

### Định giá đối chiếu chất lượng

Trục hữu ích nhất trong thực tế: rẻ **và** tốt nằm ở góc trên bên phải.

In [11]:
ve = ket_qua.dropna(subset=["Định giá", "Sinh lời"]).copy()
ve["nhan"] = ve["short_name"].str.slice(0, 18)
# Chỉ ghi tên các mã ở rìa — ghi hết 300 nhãn thì không đọc được gì
noi_bat = ve[(ve["Định giá"].abs() > 1.2) | (ve["Sinh lời"].abs() > 1.5)]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ve["Định giá"],
        y=ve["Sinh lời"],
        mode="markers",
        marker=dict(size=9, color=CHUOI[0], opacity=0.55, line=dict(width=1, color="#fcfcfb")),
        text=ve["nhan"] + " · " + ve["icb_name2"],
        hovertemplate="%{text}<br>Định giá %{x:+.2f} · Sinh lời %{y:+.2f}<extra></extra>",
        name="",
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=noi_bat["Định giá"],
        y=noi_bat["Sinh lời"],
        mode="markers+text",
        marker=dict(size=10, color=CHUOI[1], line=dict(width=1, color="#fcfcfb")),
        text=noi_bat["nhan"],
        textposition="top center",
        textfont=dict(size=10, color="#52514e"),
        hovertemplate="%{text}<extra></extra>",
        showlegend=False,
    )
)
fig.add_hline(y=0, line_width=1, line_color="#c3c2b7")
fig.add_vline(x=0, line_width=1, line_color="#c3c2b7")
fig.add_annotation(x=2.0, y=2.4, text="rẻ & sinh lời tốt", showarrow=False, font=dict(color=TANG, size=12))
fig.add_annotation(x=-2.0, y=-2.2, text="đắt & sinh lời kém", showarrow=False, font=dict(color=GIAM, size=12))
fig.update_layout(
    title_text="Định giá đối chiếu khả năng sinh lời<br>"
    "<sub style='color:#52514e'>Cả hai trục là z-score trong ngành · chỉ ghi nhãn các mã ở rìa</sub>",
    xaxis_title="Điểm định giá (cao = rẻ hơn ngành)",
    yaxis_title="Điểm sinh lời (cao = lãi tốt hơn ngành)",
    height=600,
)
fig

### Ngành nào đang có nhiều mã điểm cao

In [12]:
theo_nganh = (
    ket_qua.groupby("icb_name2", observed=True)
    .agg(diem_trung_vi=("Điểm", "median"), so_ma=("Điểm", "count"))
    .query("so_ma >= 5")
    .sort_values("diem_trung_vi", ascending=False)
    .reset_index()
)

bar_ngang(
    theo_nganh,
    nhan="icb_name2",
    gia_tri="diem_trung_vi",
    tieu_de="Điểm trung vị theo ngành",
    phu_de="Chỉ các ngành có ≥ 5 mã trong vũ trụ · nhớ rằng z-score đã chuẩn hoá TRONG ngành",
    nhan_x="điểm trung vị",
    dinh_dang_nhan="{:+.2f}",
)

⚠️ Biểu đồ này dễ bị đọc sai. Vì z-score chuẩn hoá **trong ngành**, trung vị
của mỗi ngành lẽ ra phải quanh 0. Chênh lệch còn lại đến từ: (a) các ngành nhỏ
dùng thang toàn sàn, (b) bộ lọc thanh khoản cắt bỏ phần đuôi của mỗi ngành
không đều nhau. **Nó không nói ngành nào rẻ hơn ngành nào** — thang đó đã bị
loại bỏ có chủ đích ở bước chuẩn hoá.

## 7 · Kiểm tra tỉnh táo: điểm này có nghĩa gì không?

Trước khi tin một screener, hãy hỏi nó có tương quan với thứ gì có thật không.
Đối chiếu điểm với **lợi suất 12 tháng đã qua** — không phải để chứng minh nó
dự báo được, mà để phát hiện nếu nó chỉ đơn giản là đang chọn ra các mã vừa
giảm mạnh.

In [13]:
gia_1n = client.eod.stock.ohlcv(ket_qua.index.tolist(), start=lui_ngay(HOM_NAY, nam=1))
ls_1n = (
    gia_1n.sort_values(["symbol", "date"])
    .groupby("symbol", observed=True)["close"]
    .agg(lambda s: (s.iloc[-1] / s.iloc[0] - 1) * 100)
    .rename("ls_12t")
)

kiem = ket_qua.join(ls_1n).dropna(subset=["ls_12t", "Điểm"])
print(f"Tương quan điểm tổng với lợi suất 12 tháng đã qua: {kiem['Điểm'].corr(kiem['ls_12t']):+.3f}")
for truc in TRUC:
    print(f"  {truc:<12} {kiem[truc].corr(kiem['ls_12t']):+.3f}")

Tương quan điểm tổng với lợi suất 12 tháng đã qua: -0.163
  Định giá     -0.181
  Sinh lời     +0.002
  Tăng trưởng  +0.099
  An toàn      -0.344


In [14]:
kiem["nhom_diem"] = pd.qcut(kiem["Điểm"], 5, labels=["Q1 thấp nhất", "Q2", "Q3", "Q4", "Q5 cao nhất"])
theo_nhom = (
    kiem.groupby("nhom_diem", observed=True)["ls_12t"]
    .median()
    .round(1)
    .reset_index()
)

from finlens_examples import thanh_doi_mau

thanh_doi_mau(
    theo_nhom,
    x="nhom_diem",
    y="ls_12t",
    tieu_de="Lợi suất 12 tháng ĐÃ QUA theo nhóm điểm",
    phu_de="Đây là quan hệ đồng thời, KHÔNG phải bằng chứng dự báo — điểm tính trên số liệu công bố trong kỳ đó",
    nhan_y="% trung vị",
    dinh_dang_nhan="{:+.1f}%",
)

**Đọc biểu đồ này cho đúng.** Điểm định giá được tính từ P/E và P/B của *hôm
nay*; giá hôm nay nằm ở tử số của cả hai. Một mã vừa giảm 40% tự động trở thành
"rẻ" theo thang này. Nên nhóm điểm cao có lợi suất quá khứ thấp là chuyện **cơ
học**, không phải phát hiện.

Muốn biết screener có dự báo được không thì phải backtest đúng cách: chấm điểm
bằng dữ liệu **tại thời điểm quá khứ**, rồi đo lợi suất **sau** đó. Notebook
`34` dựng bộ khung tránh look-ahead cho việc này.

## 8 · Xuất kết quả

In [15]:
THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)

xuat = ket_qua[COT_HIEN].round(3).reset_index()
tep = THU_MUC_RA / f"screener_dinh_gia_{NAM}_{HOM_NAY:%Y%m%d}.xlsx"
xuat.to_excel(tep, index=False, sheet_name=f"Xep hang {NAM}")
print(f"Đã ghi {len(xuat)} dòng → {tep.name}")

Đã ghi 140 dòng → screener_dinh_gia_2025_20260811.xlsx


## Tổng kết

| Bước | Vì sao |
|---|---|
| Lọc thanh khoản trước | định giá của mã không giao dịch là giá của thị trường không tồn tại |
| Kỳ báo cáo hỏi từ dữ liệu | notebook không chết vào tháng 3 năm sau |
| `higher_is_better` từ danh mục | dấu là dữ liệu, không phải trí nhớ |
| Cắt đuôi trước z-score | một mã P/B 47 lần đè bẹp cả thang của ngành |
| Ngành < 8 mã dùng thang toàn sàn | z-score trên 3 quan sát là nhiễu |
| `NaN` giữ nguyên, không `fillna(0)` | P/E rỗng nghĩa là lỗ, không phải rẻ |
| Yêu cầu ≥ 3 trục có dữ liệu | hai trục không đủ để gọi là đánh giá |

**Và điều quan trọng nhất:** điểm cao nghĩa là "đáng đọc báo cáo tài chính",
không phải "nên mua". Screener thu hẹp danh sách từ 400 xuống 20; hai mươi mã
đó vẫn cần được đọc từng cái một.

---

**Tiếp theo:** [`24_deep_dive_ngan_hang.ipynb`](24_deep_dive_ngan_hang.ipynb) —
bộ chỉ tiêu riêng của ngành ngân hàng, thứ mà screener chung ở trên không chạm
tới được.